# Search-12a — Composer des regards : play forward, coplay backward

**Navigation** : [Search-11d](Search-11d-Descente-Sous-Budget.ipynb) | [Python](Search-12a-Composer-Regards.ipynb) | [Search-13a](Search-13a-Traverser-Murs-Certifies.ipynb)

**Objectifs** : composer distances avant et arrière, distinguer associativité et exactitude, et comparer deux guides. **Prérequis** : graphes, Dijkstra et A*. **Durée estimée** : 25 minutes. Implémentation pédagogique BCL.

## 1. Terrain pondéré

Grille déterministe 10 × 14 : plaine 1, forêt 3, marais 8. Une arête paie la cellule dans laquelle elle entre. La figure montre les coûts et les deux extrémités.

Les deux gloutons mesurés plus loin font chacun 22 pas sur ce terrain, avec des coûts différents.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Net;
using Microsoft.DotNet.Interactive.Formatting;
record HtmlSvg(string Content);
System.Globalization.CultureInfo.DefaultThreadCurrentCulture=System.Globalization.CultureInfo.InvariantCulture;
Formatter.Register<HtmlSvg>((v,w)=>w.Write(v.Content), "text/html");
Console.WriteLine($"Runtime {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}; BCL");

Runtime .NET 10.0.14; BCL


Les coordonnées sont données par `(ligne, colonne)` ; le départ est en haut à gauche et le but en bas à droite.

In [2]:
static class Terrain {
 public const int H=10,W=14;
 public static readonly (int r,int c) Start=(0,0), Goal=(9,13);
 public static int[,] Cout=Construire();
 static int[,] Construire(){var a=new int[H,W]; for(int r=0;r<H;r++) for(int c=0;c<W;c++) a[r,c]=3; for(int r=0;r<H;r++) a[r,1]=1; for(int c=1;c<W-1;c++) a[H-2,c]=1; for(int r=H-2;r>=0;r--) a[r,W-2]=1; for(int i=2;i<Math.Min(W,H)-1;i++){a[i,i+2]=8; if(i+3<W)a[i,i+3]=8;} a[0,0]=1;a[H-1,W-1]=1;return a;}
 public static IEnumerable<(int r,int c)> Voisins((int r,int c) u){foreach(var (dr,dc) in new[]{(1,0),(-1,0),(0,1),(0,-1)}){var v=(r:u.r+dr,c:u.c+dc); if(v.r>=0&&v.r<H&&v.c>=0&&v.c<W)yield return v;}}
 public static Dictionary<(int r,int c),int> Dijkstra((int r,int c) source,bool inverse=false, Dictionary<(int r,int c),(int r,int c)> parents=null){
  var d=new Dictionary<(int,int),int>{{source,0}}; var q=new PriorityQueue<(int r,int c),(int,int,int)>();q.Enqueue(source,(0,source.r,source.c));
  while(q.TryDequeue(out var u,out var pr)){int du=pr.Item1;if(du!=d[u])continue;foreach(var v in Voisins(u)){int nd=du+(inverse?Cout[u.r,u.c]:Cout[v.r,v.c]); if(!d.ContainsKey(v)||nd<d[v]){d[v]=nd;q.Enqueue(v,(nd,v.r,v.c));if(parents!=null)parents[v]=u;}}}return d;
 }
 public static Dictionary<(int r,int c),int> BFS((int r,int c) source){var d=new Dictionary<(int,int),int>{{source,0}};var q=new Queue<(int r,int c)>();q.Enqueue(source);while(q.Count>0){var u=q.Dequeue();foreach(var v in Voisins(u))if(!d.ContainsKey(v)){d[v]=d[u]+1;q.Enqueue(v);}}return d;}
 public static List<(int r,int c)> Glouton(Dictionary<(int r,int c),int> h,bool cout){var p=new List<(int r,int c)>{Start};while(p.Last()!=Goal){var v=Voisins(p.Last()).OrderBy(v=>h[v]+(cout?Cout[v.r,v.c]:0)).First();p.Add(v);if(p.Count>H*W)throw new Exception("cycle");}return p;}
 public static int Route(List<(int r,int c)> p)=>p.Skip(1).Sum(v=>Cout[v.r,v.c]);
 public static (int cout,int expansions) AStar(Dictionary<(int r,int c),int> h){var ferme=new HashSet<(int,int)>();var q=new PriorityQueue<((int r,int c) u,int g),(int,int,int,int)>();q.Enqueue((Start,0),(h[Start],0,0,0));int ext=0;while(q.TryDequeue(out var item,out var priority)){var (u,g)=item;if(!ferme.Add(u))continue;ext++;if(u==Goal)return (g,ext);foreach(var v in Voisins(u))if(!ferme.Contains(v)){int ng=g+Cout[v.r,v.c];q.Enqueue((v,ng),(ng+h[v],ng,v.r,v.c));}}throw new Exception("but inaccessible");}
}
Console.WriteLine("Terrain et recherches BCL initialises");

Terrain et recherches BCL initialises


### Carte des coûts

Les nombres indiquent le coût d'entrée ; S est le départ et G l'arrivée. Le fond blanc fait partie de l'image.

In [3]:
var svg=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='600' viewBox='0 0 760 600'><rect width='760' height='600' fill='white'/><text x='380' y='25' text-anchor='middle' font-size='20'>Gridworld : coût d’entrée (unités)</text>");
for(int r=0;r<Terrain.H;r++)for(int col=0;col<Terrain.W;col++){int cost=Terrain.Cout[r,col];int x=80+col*43,y=60+r*43;string fill=cost==1?"#d6eec8":cost==3?"#8dc982":"#607843";string label=(r,col)==Terrain.Start?"S":(r,col)==Terrain.Goal?"G":cost.ToString(System.Globalization.CultureInfo.InvariantCulture);svg.Append($"<rect x='{x}' y='{y}' width='43' height='43' fill='{fill}' stroke='#333'/><text x='{x+21}' y='{y+27}' text-anchor='middle' font-size='15' fill='{(cost==8?"white":"black")}'>{WebUtility.HtmlEncode(label)}</text>");}
for(int i=0;i<Terrain.W;i++) svg.Append($"<text x='{101+i*43}' y='510' text-anchor='middle' font-size='11'>{i.ToString(System.Globalization.CultureInfo.InvariantCulture)}</text>");
for(int i=0;i<Terrain.H;i++) svg.Append($"<text x='70' y='{87+i*43}' text-anchor='end' font-size='11'>{i.ToString(System.Globalization.CultureInfo.InvariantCulture)}</text>");
svg.Append("<text x='380' y='540' text-anchor='middle'>Colonne</text><text x='18' y='270' transform='rotate(-90 18 270)' text-anchor='middle'>Ligne</text><text x='380' y='575' text-anchor='middle'>Clair : 1 | moyen : 3 | foncé : 8 | S : départ | G : arrivée</text></svg>");
display(new HtmlSvg(svg.ToString())); Console.WriteLine("Carte 10 x 14 : 140 cellules");

Gridworld : coût d’entrée (unités) S 1 3 3 3 3 3 3 3 3 3 3 1 3 3 1 3 3 3 3 3 3 3 3 3 3 1 3 3 1 3 3 8 8 3 3 3 3 3 3 1 3 3 1 3 3 3 8 8 3 3 3 3 3 1 3 3 1 3 3 3 3 8 8 3 3 3 3 1 3 3 1 3 3 3 3 3 8 8 3 3 3 1 3 3 1 3 3 3 3 3 3 8 8 3 3 1 3 3 1 3 3 3 3 3 3 3 8 8 3 1 3 3 1 1 1 1 1 1 1 1 1 8 8 1 3 3 1 3 3 3 3 3 3 3 3 3 3 3 G 0 1 2 3 4 5 6 7 8 9 10 11 12 13 0 1 2 3 4 5 6 7 8 9 Colonne Ligne Clair : 1 | moyen : 3 | foncé : 8 | S : départ | G : arrivée

Carte 10 x 14 : 140 cellules


## 2. Composition avant et arrière

**Play forward** : `g(n)` est le coût exact accumulé du départ à `n`. **Coplay backward** : `d(n)` est le coût exact restant de `n` au but. Leur composé `g(n)+d(n)` est le coût minimum d'une route passant par `n`.

La recherche arrière emprunte les arcs inversés en gardant leur coût original : depuis `u` vers `v` en sens inverse, on paie le coût de `u`, destination de l'arc original.

In [4]:
var g=Terrain.Dijkstra(Terrain.Start); var d=Terrain.Dijkstra(Terrain.Goal,true); int opt=g[Terrain.Goal];
int erreurs=g.Keys.Count(n=>Terrain.Dijkstra(n)[Terrain.Goal]!=d[n]);
int incoherences=g.Keys.Sum(u=>Terrain.Voisins(u).Count(v=>d[u]>Terrain.Cout[v.r,v.c]+d[v]));
if(erreurs!=0 || incoherences!=0 || d[(8,13)]!=1 || Terrain.Dijkstra(Terrain.Goal)[(8,13)]!=3)throw new Exception("distance arriere");
var corridor=g.Keys.Where(n=>g[n]+d[n]==opt).ToList(); if(g.Keys.Any(n=>g[n]+d[n]<opt))throw new Exception("composition");
Console.WriteLine($"OPT={opt} ; champs {g.Count}/{d.Count} ; erreurs oracle={erreurs}, incoherences={incoherences} ; fixture (8,13) restant={d[(8,13)]}");
Console.WriteLine($"f >= OPT : vrai ; corridor {corridor.Count}/{g.Count} noeuds");

OPT=30 ; champs 140/140 ; erreurs oracle=0, incoherences=0 ; fixture (8,13) restant=1


f >= OPT : vrai ; corridor 23/140 noeuds


La recherche depuis chaque nœud vers le but vérifie les 140 distances inverses. L'égalité `g+d=OPT` décrit les nœuds de routes optimales ; construire les champs eux-mêmes requiert des recherches sur le graphe.

Sur ce terrain le corridor contient 23 nœuds sur 140 ; OPT vaut 30.

### Exercice 1 — projection relaxante

Projeter sur les colonnes avec un coût de transition au plus égal au minimum d'entrée de la colonne cible ; les mouvements verticaux deviennent gratuits. Une restriction à une colonne est au contraire susceptible de surestimer. Retourner une borne par cellule, puis comparer ses expansions A* au guide exact.

In [5]:
static Dictionary<(int,int),int> Projection(int[,] cout,(int,int) but){
 // TODO : construire une vraie relaxation et retourner une borne par cellule.
 return null;
}
var miniCouts=new int[,]{{1,9,1},{1,1,1}};
var projection=Projection(Terrain.Cout,Terrain.Goal); var mini=Projection(miniCouts,(0,2));
var casesMini=Enumerable.Range(0,2).SelectMany(r=>Enumerable.Range(0,3).Select(c=>(r,c))).ToArray();
bool miniValide=mini is not null && mini.Keys.ToHashSet().SetEquals(casesMini) && mini.Values.All(x=>x>=0) &&
 casesMini.All(u=>new[]{(1,0),(-1,0),(0,1),(0,-1)}.Select(delta=>(r:u.r+delta.Item1,c:u.c+delta.Item2))
 .Where(v=>v.r>=0&&v.r<2&&v.c>=0&&v.c<3).All(v=>mini[u]<=miniCouts[v.r,v.c]+mini[v]));

if(projection is null || mini is null)Console.WriteLine("Exercice 1 a completer");
else if(projection.Count!=140 || g.Keys.Any(n=>!projection.ContainsKey(n)||projection[n]<0||projection[n]>d[n]) || projection[Terrain.Goal]!=0 || g.Keys.Any(u=>Terrain.Voisins(u).Any(v=>projection[u]>Terrain.Cout[v.r,v.c]+projection[v])) || !miniValide || mini[(0,0)]!=2 || mini[(1,0)]!=2 || mini[(0,2)]!=0)Console.WriteLine("Exercice 1 : controle echoue");
else {
 var guide=Terrain.AStar(projection);var exactGuide=Terrain.AStar(d);
 if(guide.cout!=opt)throw new Exception("projection coherente : cout A* incorrect");
 Console.WriteLine($"Exercice 1 : guide non negatif, admissible et coherent sur 140 noeuds ; cout {guide.cout}, expansions projection {guide.expansions}, exact {exactGuide.expansions}");
}

Exercice 1 a completer


## 3. Relais : associativité et exactitude

L'addition des tronçons est associative. L'égalité avec la distance directe exige que les relais soient sur une route optimale pour le nœud considéré.

In [6]:
var parents=new Dictionary<(int r,int c),(int r,int c)>();Terrain.Dijkstra(Terrain.Start,false,parents);var chem=new List<(int r,int c)>{Terrain.Goal};while(chem.Last()!=Terrain.Start)chem.Add(parents[chem.Last()]);chem.Reverse();
var m1=chem[chem.Count/3];var m2=chem[2*chem.Count/3];var on=chem[chem.Count/6];var off=(3,4);
Console.WriteLine($"chemin optimal : {chem.Count-1} pas, cout {Terrain.Route(chem)}");
foreach(var (nom,n) in new[]{("sur route",on),("hors route",off)}) {int a=Terrain.Dijkstra(n)[m1],b=Terrain.Dijkstra(m1)[m2],c=Terrain.Dijkstra(m2)[Terrain.Goal],direct=Terrain.Dijkstra(n)[Terrain.Goal];Console.WriteLine($"{nom} {n} : assoc {(a+b)+c==a+(b+c)}, chaine {a+b+c}, direct {direct}, ecart {a+b+c-direct}");}

chemin optimal : 22 pas, cout 30


sur route (2, 1) : assoc True, chaine 27, direct 27, ecart 0


hors route (3, 4) : assoc True, chaine 33, direct 31, ecart 2


Les deux calculs montrent que changer les parenthèses ne change pas la somme. L'écart hors route mesure le détour imposé ; être hors d'une route choisie ne suffit pas toujours à prouver un écart strict.

## 4. Guides coût et pas

Le glouton coût choisit `coût(v)+d(v)`, conformément à Bellman. Le glouton en pas minimise la distance BFS. A* utilise chacun comme heuristique ; les pas sont admissibles car chaque entrée coûte au moins 1.

In [7]:
var pas=Terrain.BFS(Terrain.Goal); var pc=Terrain.Glouton(d,true);var ps=Terrain.Glouton(pas,false);int div=Enumerable.Range(0,Math.Min(pc.Count,ps.Count)).First(i=>pc[i]!=ps[i]);
var ac=Terrain.AStar(d);var ap=Terrain.AStar(pas);if(ac.cout!=opt||ap.cout!=opt||g.Keys.Any(n=>pas[n]>d[n]))throw new Exception("A* ou admissibilite");
Console.WriteLine($"glouton cout {pc.Count-1} pas, cout {Terrain.Route(pc)} ; pas {ps.Count-1} pas, cout {Terrain.Route(ps)} ; divergence {div} : {pc[div]} vs {ps[div]}");
Console.WriteLine($"A* exact : {ac.cout}, {ac.expansions} expansions ; A* pas : {ap.cout}, {ap.expansions} expansions ; admissibilite 140/140");

glouton cout 22 pas, cout 30 ; pas 22 pas, cout 62 ; divergence 1 : (0, 1) vs (1, 0)


A* exact : 30, 23 expansions ; A* pas : 30, 75 expansions ; admissibilite 140/140


Les deux guides A* rendent ici l'optimum mais ne portent pas la même information. Les expansions dépendent du départage. Le glouton en pas peut payer plus cher tout en empruntant le même nombre de pas.

Ici, les deux gloutons font 22 pas mais coûtent 30 et 62 ; les deux A* effectuent respectivement 23 et 75 expansions.

### Exercice 2 — A* pondéré

Pour ε = 1, 1,5, 2, 3, utiliser `f=g+εh`, départager par `(f,g,ligne,colonne)` et appeler `expansion(noeud,g)` pour chaque nœud fermé, but compris. Retourner coût et chemin ; le harnais rejoue le front et compte les expansions observées. L'heuristique est un paramètre : les fixtures utilisent un guide nul puis un guide exact. Ne pas supposer une perte d'optimalité sur ce terrain.

In [8]:
#nullable enable
record CheminPondere(int Cout,List<(int r,int c)> Chemin);
static CheminPondere? Pondere(double epsilon, Dictionary<(int r,int c),int> h,int[,] couts,
 (int r,int c) depart,(int r,int c) but,Action<(int r,int c),int> expansion){
 // TODO : appeler expansion(noeud,g) pour chaque noeud ferme, but compris.
 // Utiliser f=g+epsilon*h et departager par f,g,ligne,colonne.
 return null;
}
static (int cout,int expansions)? VerifierPondere(double epsilon,Dictionary<(int r,int c),int> h,int[,] couts,
 (int r,int c) depart,(int r,int c) but){
 int hauteur=couts.GetLength(0),largeur=couts.GetLength(1);var fermes=new HashSet<(int r,int c)>();
 var ouverts=new PriorityQueue<((int r,int c) u,int g),(double f,int g,int r,int c)>();
 ouverts.Enqueue((depart,0),(epsilon*h[depart],0,depart.r,depart.c));int compte=0,coutBut=-1;bool arrive=false;
 void Expansion((int r,int c) u,int gu){
  if(arrive || compte>=hauteur*largeur)throw new Exception("expansion apres but ou repetition");
  while(ouverts.TryPeek(out var top,out _) && fermes.Contains(top.u))ouverts.Dequeue();
  if(!ouverts.TryDequeue(out var attendu,out _) || attendu.u!=u || attendu.g!=gu)throw new Exception("ordre f,g,ligne,colonne");
  fermes.Add(u);compte++;if(u==but){arrive=true;coutBut=gu;return;}
  foreach(var (dr,dc) in new[]{(1,0),(-1,0),(0,1),(0,-1)}){
   var v=(r:u.r+dr,c:u.c+dc);if(v.r>=0 && v.r<hauteur && v.c>=0 && v.c<largeur && !fermes.Contains(v)){
    int gv=gu+couts[v.r,v.c];ouverts.Enqueue((v,gv),(gv+epsilon*h[v],gv,v.r,v.c));
   }
  }
 }
 var r=Pondere(epsilon,h,couts,depart,but,Expansion);if(r is null)return null;
 if(!arrive || r.Cout!=coutBut || r.Chemin is null || r.Chemin.Count==0 || r.Chemin[0]!=depart || r.Chemin.Last()!=but ||
    r.Chemin.Any(u=>u.r<0 || u.r>=hauteur || u.c<0 || u.c>=largeur) ||
    r.Chemin.Zip(r.Chemin.Skip(1),(u,v)=>Math.Abs(u.r-v.r)+Math.Abs(u.c-v.c)).Any(distance=>distance!=1) ||
    r.Chemin.Skip(1).Sum(u=>couts[u.r,u.c])!=r.Cout)throw new Exception("chemin ou cout invalide");
 return (r.Cout,compte);
}
var ep=new[]{1.0,1.5,2.0,3.0};var essais=ep.Select(e=>VerifierPondere(e,d,Terrain.Cout,Terrain.Start,Terrain.Goal)).ToArray();
var miniP=new int[,]{{1,2},{4,1}};var hZero=new Dictionary<(int r,int c),int>{{(0,0),0},{(0,1),0},{(1,0),0},{(1,1),0}};
var hParfait=new Dictionary<(int r,int c),int>{{(0,0),4},{(0,1),5},{(1,0),0},{(1,1),4}};
var petit1=VerifierPondere(1,hZero,miniP,(0,0),(1,1));var petit2=VerifierPondere(1,hZero,miniP,(0,0),(1,0));
var parfait=VerifierPondere(1,hParfait,miniP,(0,0),(1,0));
if(essais.Any(x=>x is null) || petit1 is null || petit2 is null || parfait is null)Console.WriteLine("Exercice 2 a completer");
else if(essais[0]!.Value.cout!=opt || essais.Any(x=>x!.Value.cout<opt) || petit1.Value!=(3,3) || petit2.Value!=(4,4) || parfait.Value!=(4,2))throw new Exception("fixtures ponderees");
else Console.WriteLine("Exercice 2 — (cout, expansions observees) : " + string.Join(", ",essais));


Exercice 2 a completer


### Exercice 3 — métrique uniforme

Reconstruire une grille dont tous les coûts valent un, rejouer le guide en pas et Dijkstra, puis retourner leurs coûts. Deux dimensions vérifient que la réponse dépend du terrain.

In [9]:
static (int glouton,int optimal)? Uniforme(int hauteur,int largeur){
 // TODO : construire la grille et rejouer les recherches.
 return null;
}
var u1=Uniforme(10,14);var u2=Uniforme(3,4);
if(u1 is null || u2 is null)Console.WriteLine("Exercice 3 a completer");
else Console.WriteLine(u1.Value==(22,22) && u2.Value==(5,5) ? "Exercice 3 : fixtures valides" : "Exercice 3 : controle echoue");

Exercice 3 a completer


## 5. Tableau de synthèse

Les trois lignes résument la composition coût/coût, les relais et la différence des deux guides.

In [10]:
int Leg((int r,int c) a,(int r,int c) b)=>Terrain.Dijkstra(a)[b];
int chaineOn=Leg(on,m1)+Leg(m1,m2)+Leg(m2,Terrain.Goal), directOn=Leg(on,Terrain.Goal);
int chaineOff=Leg(off,m1)+Leg(m1,m2)+Leg(m2,Terrain.Goal), directOff=Leg(off,Terrain.Goal);
Console.WriteLine("paire | unites | garantie | temoin");
Console.WriteLine($"cout avant + cout arriere | coherentes | corridor optimal | {corridor.Count}/{g.Count} noeuds, OPT {opt}");
Console.WriteLine($"cout + relais + cout | coherentes | associativite, exactitude si relais sur une route optimale | {chaineOn}/{directOn} sur route, {chaineOff}/{directOff} hors route");
Console.WriteLine($"cout avant + pas arriere | differentes | borne admissible, pas distance exacte en cout | divergence {div}, couts {Terrain.Route(pc)}/{Terrain.Route(ps)}, A* {ac.expansions}/{ap.expansions} expansions");

paire | unites | garantie | temoin


cout avant + cout arriere | coherentes | corridor optimal | 23/140 noeuds, OPT 30


cout + relais + cout | coherentes | associativite, exactitude si relais sur une route optimale | 27/27 sur route, 33/31 hors route


cout avant + pas arriere | differentes | borne admissible, pas distance exacte en cout | divergence 1, couts 30/62, A* 23/75 expansions


## À retenir

Inverser un arc conserve son coût ; cela rend les distances restantes exactes. `g+d=OPT` caractérise le corridor optimal. Les relais préservent l'associativité, mais peuvent introduire un détour. Le guide en pas est admissible sans être exact.

## Vérifiez votre compréhension

<details><summary>Quel coût paie l'arc inversé ?</summary>Celui de la destination de l'arc original, donc de la source de l'arc inversé.</details>

<details><summary>Quel est le choix glouton correct ?</summary>Minimiser le coût de l'arc suivant plus la distance restante.</details>

## Pour aller plus loin

[Search-03e — Optimalité d'A*](Search-03e-AStar-Optimality.ipynb) et [Search-03b — Pattern databases](Search-03b-PatternDatabases.ipynb). Les trois exercices portent sur la projection, la pondération et la métrique uniforme.